### 回帰モデルに対する事後分布と事後統計量

$$ x_i = 
\begin{pmatrix}
    x_{1i} \\ \vdots \\ x_{ki}
\end{pmatrix}

\boldsymbol{\beta} = 
\begin{pmatrix}
    \beta_1 \\ \vdots \\ \beta_k
\end{pmatrix}
$$

```math 
\boldsymbol{y} = 
\begin{pmatrix}
    y_1 \\ \vdots \\ y_n
\end{pmatrix}

\boldsymbol{X} = 
\begin{pmatrix}
    x_{11} \cdots x_{ki} \\
    \vdots \ddots \vdots \\
    x_{1n} \cdots x_{kn}
\end{pmatrix}

= \begin{pmatrix}
    \boldsymbol{x_1^T} \\
    \vdots \\
    \boldsymbol{x_n^T}
\end{pmatrix}

\boldsymbol{u} = 
\begin{pmatrix}
    u_1 \\ \vdots \\ u_n
\end{pmatrix}
```

$$ \boldsymbol{y} = \boldsymbol{X}\boldsymbol{\beta} + \boldsymbol{u} $$
$$ \boldsymbol{u} \sim N_{n}(\boldsymbol{0}, \sigma^2\boldsymbol{I}) $$
$$ N_{n}(\boldsymbol{0}, \sigma^2\boldsymbol{I}) = p(\boldsymbol{x}|\boldsymbol{\mu},\boldsymbol{\Sigma}) = (2\pi)^{-\frac{m}{2}} |\Sigma|^{-\frac{1}{2}} exp[-\frac{1}{2}(\boldsymbol{x}-\boldsymbol{\mu})^T\Sigma^{-1}(\boldsymbol{x}-\boldsymbol{\mu})] $$

$$ E[y_i|\boldsymbol{x_i}] = \beta_1x_{1i} + \beta_2x_{2i} + \cdots + \beta_kx_{ki} $$

$$ \boldsymbol{\beta}|\sigma^2 \sim N_k(\boldsymbol{\beta}_0, \sigma^2\boldsymbol{A}_{0}^{-1}) $$
$$ \sigma^2 \sim Ga^{-1}(\frac{\nu_0}{2},\frac{\lambda_0}{2}) $$

#### 回帰モデルの同時事前分布(確率密度)

##### 精度行列 $ \boldsymbol{A}_0 $
```math
\boldsymbol{A_0} = 
\begin{pmatrix}
    cov_{11}^{-1} \cdots cov_{1k}^{-1} \\
    \vdots \ddots \vdots \\
    cov_{n1}^{-1} \cdots cov_{nk}^{-1}  
\end{pmatrix}
```

##### 事前分布
$$ p(\beta, \sigma^2) = p(\beta|\sigma^2)p(\sigma^2) $$
$$ p(\beta|\sigma^2) = (2\pi\sigma^2)^{-\frac{k}{2}}|\boldsymbol{A}_0|^{\frac{1}{2}} exp[-\frac{1}{2\sigma^2}(\boldsymbol{\beta} - \boldsymbol{\beta}_0)^T \boldsymbol{A}_0 (\boldsymbol{\beta} - \boldsymbol{\beta}_0)] $$
$$ p(\sigma^2) = \frac{(\frac{\lambda_0}{2})^{\frac{\nu_0}{2}}}{\Gamma(\frac{\nu_0}{2})}(\sigma^2)^{-\frac{\nu_0}{2} + 1} exp[-\frac{\lambda_0}{2\sigma^2}] $$ 

### 回帰モデルの尤度

$$  $$

### 回帰モデルの事後分布

$$ \boldsymbol{\beta}|\sigma^2 , D \sim N_{K}(\boldsymbol{\beta}_{*}, \sigma^2\boldsymbol{A}_{*}^{-1}) $$
$$ \sigma^2 | D \sim Ga^{-1}(\frac{\nu_{*}}{2}, \frac{\lambda_{*}}{2}) $$

#### 回帰モデルの事後分布に関する計算式
+ $ \boldsymbol{\beta}_{*} = (\boldsymbol{X}^T\boldsymbol{X}+\boldsymbol{A}_0)^{-1}(\boldsymbol{X}^T\boldsymbol{y}+\boldsymbol{A}_0\boldsymbol{\beta}_0) $
+ $ \boldsymbol{A}_{*} = \boldsymbol{X}^T\boldsymbol{X} + \boldsymbol{A}_0 $
+ $ \nu_{*} = n + \nu_0 $
+ $ \lambda_{*} = \boldsymbol{(y - \hat{y})}^T\boldsymbol{(\boldsymbol{y - \hat{y}})} + (\boldsymbol{\beta - \hat{\beta}})\boldsymbol{C}_{*} (\boldsymbol{\beta - \hat{\beta}}) + \lambda_0 $
+ $ \hat{\boldsymbol{y}} = \boldsymbol{X}\hat{\boldsymbol{\beta}} $ 最小二乗法による推定値
+ $ \hat{\boldsymbol{\beta}} = (\boldsymbol{X}^T\boldsymbol{X})^{-1}\boldsymbol{X}^{T}\boldsymbol{y} $
+ $ \boldsymbol{C}_{*} = ((\boldsymbol{X}^T\boldsymbol{X})^{-1} + \boldsymbol{A}_{0}^{-1})^{-1} $

### $ \beta $ に関する $ \sigma^2 $ に依存しない事後分布
$$ \boldsymbol{\beta}|D \sim T_{k}(\nu_{*}, \boldsymbol{\beta}_{*}, \boldsymbol{H}_*) $$

In [1]:
import numpy as np
np.set_printoptions(precision=3)
import scipy as sp
import scipy.stats as st
import scipy.optimize as opt
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
# import pymc
import psutil
import pandas as pd

In [2]:
import scipy.linalg as la

In [3]:
# 逆ガンマ分布のHPD区間の計算
def invgamma_hpdi(hpdi0, alpha, beta, prob):
    """入力

    Args:
        hpdi0 (_type_): HPD区間の初期値
        alpha (_type_): 形状パラメータ
        beta (_type_): 尺度パラメータ
        prob (_type_): 1 - c
    """

    def hpdi_conditions(v, a, b, p):
        # 確率
        eq1 = st.invgamma.cdf(v[1], a, scale=b) - st.invgamma.cdf(v[0], a, scale=b) - p
        # 確率密度
        eq2 = st.invgamma.pdf(v[1], a, scale=b) - st.invgamma.pdf(v[0], a, scale=b)

        return np.hstack((eq1, eq2))
    
    return opt.root(hpdi_conditions, hpdi0, args=(alpha, beta, prob)).x

In [ ]:
# 回帰モデルの係数と誤差項の分散の事後統計量の計算
def regression_stats(y: np.ndarray, 
                     X: np.ndarray, 
                     b0: np.ndarray, 
                     A0: np.ndarray, 
                     nu0: float, 
                     lam0: float, 
                     prob: float):
    """
    入力
    Args:
        y (np.ndarray): _description_
        X (np.ndarray): _description_
        b0 (np.ndarray): 回帰係数の条件付き事前分布(多変量正規分布)の平均
        A0 (np.ndarray): 回帰係数の条件付き事前分布(多変量正規分布)の精度行列
        nu0 (float): 誤差項の分散の事前分布(逆ガンマ分布)の形状パラメータ
        lam0 (float): 誤差項の分散の事前分布(逆ガンマ分布)の尺度パラメータ
        prob (float): 区間確率(0 < prob < 1)

    出力
    Args:
        results:
        b_star: 
        A_star:
        nu_star:
        lam_star
    """

    XX = X.T @ X # (K,N) @ (N,K) = (K,K)
    Xy = X.T @ y # (K,N) @ (N,) = (K,)
    b_hat = la.solve(XX, Xy) # Ax=b 連立方程式の解x
    A_star = XX + A0 # (K,K) + (K,K) = (K,K)
    b_star = la.solve(A_star, Xy + A0 @ b0)
    C_star = la.inv(la.inv(XX) + la.inv(A0))
    nu_star = y.size + nu0
    lam_star = np.square(y - X @ b_hat).sum() + (b0 - b_hat).T @ C_star @ (b0 - b_hat) + lam0
    h_star = np.sqrt(lam_star / nu_star * np.diag(la.inv(A_star))) # H* の対角要素

    # 多変量t分布
    sd_b = st.t.std(nu_star, loc=b_star, scale=h_star)

    # 信頼区間(a,b)
    ci_b = np.vstack(st.t.interval(prob, nu_star, loc=b_star, scale=h_star))
    hpdi_b = ci_b
    
    stats_b = np.vstack((b_star, b_star, b_star, sd_b, ci_b, hpdi_b)).T
    
